# Trajectory analysis with PILOT
<hr style="border:2px solid black"> </hr>

In [ ]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[3, 5, 6, 10, 12, 15, 17, 19, 22, 24, 26])

nb_name = "PILOT_trajectory.ipynb"

_compare_version(nb_name)

## 1 -  Description

**Welcome to our extented notebook for scRNA/pathomics Data for unsupervised trajectory analysis with PILOT:**

PILOT is a python library developed by Costalab to perform patient-level analysis by using optimal transport. The objective is to enable the direct comparison between multiple scales samples e.g. control vs. disease and the observation of gene expression at different timepoints.
    
    
**Reference**: Joodaki et al. (2024) Detection of PatIent-Level distances from single cell genomics and pathomics data with   
    Optimal Transport (PILOT), Molecular Systems Biology, Volume 20 

<div class="alert alert-block alert-danger">
    
**Requires a clustered or otherwise categorized anndata object. A clustering can be generated with a clustering notebook (e.g. <code>rna_analysis/notebooks/04_clustering.ipynb</code>).**
    
**Move this notebook into the notebook folder (e.g. <code>rna_analysis/notebooks/</code>) of the respective analysis before using it!**

</div>

-----------

## 2 - Setup

### 2.1 Loading packages

In [ ]:
from sctoolbox import settings
import scanpy as sc
import pandas as pd
import pilotpy as pl
import sctoolbox
import sctoolbox.utils as utils

### 2.2 Setting input/output path

In [ ]:
# In/output paths
settings.adata_input_dir = "../adatas/"
settings.adata_output_dir = "../adatas/pilot/"
settings.figure_dir = "../figures/pilot/"
settings.table_dir = "../tables/pilot/"
settings.log_file: "../logs/pilot_log.txt"

<div class="alert alert-block alert-danger">
    
**If your anndata is already clustered and was not generated with scframework, please move or copy it to the folder <code>../adatas</code>**
    
</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
clustered_adata = "anndata_5.h5ad"

-----------

## 3 - Load anndata

In [ ]:
adata = utils.adata.load_h5ad(clustered_adata) 

with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata)
    display(adata.obs)
    display(adata.var)
    display(adata.obsm)

-----------

## 4 - General input

### 4.1 Input

<div class="alert alert-block alert-danger">

**In order to use this Notebook your andata need all requiered informations below!**

</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Provide the name of the variable in the obsm level that holds the dimension reduction (PCA representation).
emb_matrix = 'X_pca'

# Specify the name of the column that corresponds to cell types or clusters.
clusters = 'celltypes'

# Indicate the column name in the observation level of your Anndata that contains information about samples or patients.
sample  = 'Sample'

# Provide the column name that represents the status or disease (e.g., “control” or “case”).
status = 'timepoint'

### 4.2 Remove specific celltypes/cluster 

In special cases, it may be necessary to remove certain cell types/clusters (e.g., in the case of contamination by blood cells) or they may simply not be of interest. In these cases, you can use the cell below to remove them from the anndata.


<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
adata= adata[adata.obs[clusters] != "Blood - Immune cells"]
adata= adata[adata.obs[clusters] != 'Blood - red blood cells']

-----------

## 5 - Computing Wasserstein distance

To find similarities in celltypes and samples this module uses optimal transport accoring to Pyré & Cuturi. For that each sample/patient is modeled as a distribution of cells in clusters (either a specific celltype or -state). The optimal transport is then used to create the transport plan T, asssuming a mass movement between cluster pairs.
    
To explain it more clearly, here is a example of myocardial infarction which Costalab used as a reference: 
 * some cellular changes e.g. conversion of healthy cardiomyocyles into damage cardiomyocyles material represents short-    term events in the disease progression --> low costs 
 * on the other hand tissue remodeling e.g. scar formation process represent long term events --> high costs 

</div>

### 5.1 Input

In [ ]:
pl.tl.wasserstein_distance(
    adata,
    emb_matrix= emb_matrix,
    clusters_col=clusters,
    sample_col= sample,
    status= status
    )

### 5.2 Ploting the Cost matrix and the Wasserstein distance heatmaps:

After the computation of the optimal transport plan, we get the heatmaps of the cost matrix and the wasserstein distance. The transport between similar cell clusters represents a low cost (darker blue) and thus transport between more different cell clusters has higher cost (lighter blue). The similarities in total are represented with a phylogeny tree.
    
* **Costmatrix:** Here you can see the similarities between celltypes/clusters. 
    
* **Wasserstein distance:** This plot shows the similarities between the timepoints/samples. It is estimated as the total cost of the optimal transport. 

In [ ]:
pl.pl.heatmaps(adata,path="../figures/pilot/")

-----------

## 6. Plotting Diffusion map of Wasserstein distance and estimate the trajectory 

### 6.1 Diffusion map of Wasserstein distance 

This visualization displays the trajektory by using a diffuison map embedding. It takes the categories from the parameter <code>status</code>. 

In [ ]:
pl.pl.trajectory(adata, path="../figures/pilot/")

### 6.2 Estimate the timepoints 

<div class="alert alert-block alert-danger">

**which are used as the pseudotime for the following plots! For that, it is important to define the start of the trajectory with the help of the parameter <code>source_node.</code>** 
    

To do this, proceed as follows:
As you can see after starting the cell below, this function takes the diffusionmap and compute a principal graph with nodes over it (The red points are the points of the diffusionmap).After that, if you have timepoints look at the diffusionmap and find the starting point. Now look at the nodes and find the node near this point and set it as the <code>source_node.</code> In case of patients samples look where you can find the control/healthy samples, take the node at the end of the graph and set this one as the <code>source_node.</code>
    
    
**Pay attention to restart this function and all following ones if you changed the <code>source_node.</code>!**

</div>

**Parameters:**
    
*   **<code> NumNodes</code>**:The number of nodes you want to use for the principal graph. The higher the number the more the nodes nestle    on the timepoints, which can result into a better estimation.
    
    
*  **<code>source_node</code>**: The first timepoint or the patient/sample which determine the start of the trajectory.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
NumNodes = 20
source_node = 2

In [ ]:
pl.pl.fit_pricipla_graph(adata, NumNodes =NumNodes , source_node = source_node, path="../figures/pilot/")

-----------

## 7 - Cell proportion

Next, we get the cell proportion for each time point using robust regression model to find cells whose proportions change linearly or non-linearly with disease progression. While the heatmap reveals the cell proportion with the sorted pseudotime on the right, the plot below visualized it in total with the regression model.

**Description for the second plot:**
* X-axis: Cell coverage
* Y-axis: Samples = pseudotime
* Points: Samples
* Line: median trend 

<h1><center>⬐ Fill in input data here ⬎</center></h1>

<div class="alert alert-block alert-danger">
    
**Before starting this function, please decide how you want to label the x-axis by determining if you have timepoints or samples with e.g. patient id.**

* **"timepoints"**: Labels the x-axis for a better overview. 
* **"samples"**: In case of samples where it might be unnecessary to show the label, the aixs simply displays the number instead.

</div>

In [ ]:
#Determine if you have "timepoints" or "samples" 
axis = "timepoints"

In [ ]:
pl.tl.cell_importance(adata,axis=axis,path_plt="../figures/pilot/",path_table="../tables/pilot/")

-----------

## 8. Finding gene markers for the clusters

### 8.1 Gene selection 

Given that we found interesting cell types, we would like to investigate genes associated with these trajectories, i.e. genes, whose expression changes linearly or quadratically with the disease progression.

After running the command, you can find a folder named ‘Markers’ inside the 'Results_PILOT' folder. There, we will have a folder for each cell type. The file ‘Whole_expressions.csv’ contains all statistics associated with genes for that cell type.
 
Here, we run the genes_importance function for all cell types.

<div class="alert alert-block alert-danger">
    
**Note that if you are running this step on a personal computer, it might take time. For one celltype/cluster the computing need around 30-60min.**
    
</div>

In [ ]:
for cell in adata.uns['cellnames']:
    pl.tl.genes_importance(adata,
                           name_cell = cell,
                           sample_col = sample,
                           col_cell = clusters,
                           plot_genes = False,
                           path="../tables/pilot/")

### 8.2 Finding specifc marker for given cluster 

The previous test only finds genes with significant changes over time for a given cell type. However, it does not consider if a similar pattern and expression values are found in other clusters. To further select genes, we use a Wald test that compares the fit of the gene in the cluster vs. the fit of the gene in other clusters.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
#Number of top genes to consider for each expression pattern
number_genes = 10 

#clusters/celltypes of interest
cellnames = ['Endocardium','Epicardium']

In [ ]:
pl.tl.gene_cluster_differentiation(adata,cellnames = cellnames, number_genes = number_genes, path="../tables/pilot/")

### 8.3 Showing specific marker of celltype of interest

Test results are saved in ‘gene_clusters_stats_extend.csv’. To find a final list of genes, we only consider genes with a fold change higher than 0.5, i.e. genes which expression is increased in the cluster at hand; and we sort the genes based on the Wald test p-value. These can be seen below.

<div class="alert alert-block alert-danger">
    
Please set from here the cluster/celltype of interest you want to have a more further analysis in the following functions.
    
</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
#The celltype/cluster of interest 
cluster = 'Endocardium'

In [ ]:
df = pl.tl.results_gene_cluster_differentiation(cluster_name = cluster, path="../tables/pilot/").head(10)
df.head(15)

-----------

## 9. Expression pattern of specfic genes

### 9.1 Expression pattern over time 

We can visualize the expression pattern of specific genes over time for the given cluster.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

<div class="alert alert-block alert-danger">
    
**Look in previous table for the gene of interest in the column**
    
</div>

In [ ]:
#The gene of interest 
gene_list = ['tfpia','icn','srgn']

In [ ]:
pl.pl.plot_gene_list_pattern(gene_list, cluster, path="../figures/pilot/")

### 9.2 Comparing expression pattern with other clusters

In the plot below, the orange line indicates the fit in the target cell type (shown as orange lines) compared to other cell types (represented by grey lines).

In [ ]:
pl.pl.exploring_specific_genes(cluster_name = cluster, gene_list = gene_list, path="../figures/pilot/")

-----------

## 10. Go enrichment 

Here is the GO enrichment for  the 50 first top genes (FC >= 0.5 and p-value < 0.01). Plot is saved at Go folder.

<div class="alert alert-block alert-danger">

**Note that the default organsim for this function is homo sapiens. For all other ones you can use the parameter  <code>organism</code>. Since gprofiler is used for the go enrichment, you need to look up on the website below in order to find the ID for your interest organism which you are hand over to the parameter.**

    
Here are some common organism and their ID: 

| Organism  | gprofil-ID |
|--------|-------------|
|mouse|"mmusculus"|
|rat|"rnorvegicus"|
|zebrafish|"drerio"|   
    
**Organism-ID for gProfiler:<button type="button" class="btn btn-link">https://biit.cs.ut.ee/gprofiler/page/organism-list </button>**


</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
#The organism of interest 
organism = "drerio"

In [ ]:
pl.pl.go_enrichment(df, cell_type = cluster, organism = organism, path_plt="../figures/pilot/", path_table="../tables/pilot/")


-----------

## 11. Group specfic genes by pattern

Here, we cluster genes based on the pattern found for each and plot their heatmap. Below the heatmap, we depict the pattern of each group's 
genes and top 10 genes having significant changes through disease progression.

You can find curves activities' statistical scores that show the fold changes of the genes through disease progression in the Markers folder for each cell type separately.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
#Set the range of the cluster
scaler_value = 0.4

In [ ]:
pl.pl.genes_selection_analysis(adata, cluster, scaler_value = scaler_value, path_to_results= "../tables/pilot/")

-----------

## 12. Hallmarker genes 

Here, we utilize the Enrichr tools to get the hallmarks of the clustered genes. The default dataset is MSigDB_Hallmark_2020, which you can change using the <code>gene_set_library</code> parameter.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
dataset = 'MSigDB_Hallmark_2020'

In [ ]:
pl.pl.genes_selection_heatmap(adata, cluster, scaler_value = scaler_value, path_to_results= "../tables/pilot/")

In [ ]:
pl.pl.plot_hallmark_genes_clusters(adata, cluster, dataset, path="../figures/pilot/")

-----------

## 13. Saving adata

In [ ]:
utils.adata.save_h5ad(adata, "anndata_pilot.h5ad")

In [ ]:
settings.close_logfile()